<a href="https://colab.research.google.com/github/hussainiram0601-lang/Industrial-Predictive-Maintenance-System/blob/main/03_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3. Data Cleaning

This notebook handles:
1. Domain Validation(SMART bounds & values)       
2. Inconsistency Treatment(Negative & zero handling)
3. Cross-Column Consistency(Device timelines & logic)
             
> **Input**: `predictive_maintenance_preprocessed.csv`  
> **Output**: `predictive_maintenance_cleaned.csv`



In [1]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', 50)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Environment ready!')


Environment ready!


## 1. Load Preprocessed Data


In [2]:
# LOAD PREPROCESSED DATA

df = pd.read_csv('predictive_maintenance_preprocessed.csv')

# Restore datetime types
for col in ['cur_datetime', 'open_datetime']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print(f'Loaded: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

Loaded: (109362, 11)
Columns: ['date', 'device', 'failure', 'metric1', 'metric2', 'metric3', 'metric4', 'metric5', 'metric6', 'metric7', 'metric9']


,date,device,failure,metric1,metric2,metric3,metric4,metric5,metric6,metric7,metric9
0,2015-01-11,S1F01E6Y,0,154306584.0000,0.0000,0.0000,0.0000,12.0000,250284.0000,0.0000,0.0000
1,2015-02-11,S1F01R2B,0,60108664.0000,0.0000,0.0000,0.0000,17.0000,329766.0000,0.0000,3.0000
2,2015-06-09,S1F01R2B,0,108219656.0000,0.0000,0.0000,0.0000,17.0000,329866.0000,0.0000,3.0000
3,2015-01-19,S1F02L38,0,125996808.0000,0.0000,0.0000,0.0000,2.0000,239678.0000,0.0000,8.0000
4,2015-02-11,S1F02L38,0,40490200.0000,0.0000,0.0000,0.0000,2.0000,242319.0000,0.0000,8.0000


## 2. Domain-knowledge Validation


In [5]:
df = df.sort_values(by=['device', 'date']).reset_index(drop=True)

### Monotonicity Check ("Things that count up should never go down")

In [6]:
# Metrics expected to be cumulative (adjust based on your dataset)
cumulative_metrics = ['metric2', 'metric3', 'metric4', 'metric7', 'metric9']

print("--- 1. Monotonicity Violation Summary ---")
for col in cumulative_metrics:
    if col in df.columns:
        # Calculate day-over-day change per device
        daily_diff = df.groupby('device')[col].diff()

        # Count negative drops
        violations = (daily_diff < 0).sum()
        print(f"[{col}]: {violations} decreasing value instances found.")

--- 1. Monotonicity Violation Summary ---
[metric2]: 5 decreasing value instances found.
[metric3]: 0 decreasing value instances found.
[metric4]: 0 decreasing value instances found.
[metric7]: 15 decreasing value instances found.
[metric9]: 0 decreasing value instances found.


### Range & Physical Bound Checks ("No impossible numbers")

In [7]:
# All telemetry metrics should strictly be >= 0
all_metrics = [f'metric{i}' for i in range(1, 10)]

print("\n--- 2. Negative Value & Outlier Bounds ---")
for col in all_metrics:
    if col in df.columns:
        neg_count = (df[col] < 0).sum()
        min_val = df[col].min()
        max_val = df[col].max()
        print(f"[{col}]: Min = {min_val:10.2f} | Max = {max_val:10.2f} | Negative Count = {neg_count}")


--- 2. Negative Value & Outlier Bounds ---
[metric1]: Min =   -7671.00 | Max = 244140480.00 | Negative Count = 20
[metric2]: Min =       0.00 | Max =   64968.00 | Negative Count = 0
[metric3]: Min =       0.00 | Max =   24929.00 | Negative Count = 0
[metric4]: Min =       0.00 | Max =    1666.00 | Negative Count = 0
[metric5]: Min =       1.00 | Max =      98.00 | Negative Count = 0
[metric6]: Min =   -9790.00 | Max =  689161.00 | Negative Count = 27
[metric7]: Min =       0.00 | Max =     832.00 | Negative Count = 0
[metric9]: Min =       0.00 | Max =   18701.00 | Negative Count = 0


### Post-Failure Logic Check ("Dead machines don't talk")

In [8]:
print("\n--- 3. Post-Failure Ghost Record Check ---")

# Extract the exact failure date for each failed device
failure_dates = (
    df[df['failure'] == 1]
    .groupby('device')['date']
    .min()
    .reset_index()
    .rename(columns={'date': 'failure_date'})
)

# Merge failure date back into the main dataset
df = df.merge(failure_dates, on='device', how='left')

# Identify rows where the record date is AFTER the device's failure date
ghost_mask = df['failure_date'].notna() & (df['date'] > df['failure_date'])
ghost_count = ghost_mask.sum()

print(f"Found {ghost_count} ghost records recorded after device failure.")

# Clean ghost records
if ghost_count > 0:
    df = df[~ghost_mask].reset_index(drop=True)
    print(f"Cleaned dataset size: {len(df)} records remaining.")

# Drop helper column
df = df.drop(columns=['failure_date'])


--- 3. Post-Failure Ghost Record Check ---
Found 45 ghost records recorded after device failure.
Cleaned dataset size: 109317 records remaining.


### Duplicate Log Check ("One diary entry per day")

In [9]:
print("\n--- 4. Uniqueness & Duplicate Check ---")

duplicates = df.duplicated(subset=['device', 'date'], keep=False)
dup_count = duplicates.sum()

print(f"Duplicate (device, date) rows found: {dup_count}")

# Remove duplicates if found (keeping the first occurrence)
if dup_count > 0:
    df = df.drop_duplicates(subset=['device', 'date'], keep='first').reset_index(drop=True)
    print("Duplicates removed successfully.")


--- 4. Uniqueness & Duplicate Check ---
Duplicate (device, date) rows found: 0
